## Environment prep

In [ ]:
!pip install hampel

In [ ]:
from hampel import hampel
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
import os
import matplotlib.pyplot as plt
import re
from math import sqrt, atan2

## Load all data into a list of dataframes

In [ ]:
# Specify the path to your main folder
main_folder_path = 'c:/Users/Ayush/Desktop/New folder (2)/ESP32-WiFi-Sensing-master/datasets/01-tvat-raw'

# Initialize an empty dictionary to store lists of DataFrames for each activity
activity_dataframes = []
label_list = []

# Iterate through subfolders
for activity_folder in os.listdir(main_folder_path):
    activity_folder_path = os.path.join(main_folder_path, activity_folder)
    # Check if it's a directory
    if os.path.isdir(activity_folder_path):
        # Iterate through files in the subfolder
        for file_name in os.listdir(activity_folder_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(activity_folder_path, file_name)
                label_list.append(activity_folder)
                # Read the data from each CSV file into a DataFrame
                df = pd.read_csv(file_path)
                activity_dataframes.append(df)


In [ ]:
print(len(activity_dataframes))

## Extract amplitude from dataframes

In [ ]:
amp_dataframes = []
# Loop through each file
for df in activity_dataframes:
    # Construct the full path to the file
    data = []
    for _, row in df.iterrows():
      imaginary = []
      real = []
      amplitudes = []

        # Parse string to create integer list
      csi_string = re.findall(r"\[(.*)\]", row['CSI_DATA'])[0]
      csi_raw = [int(x) for x in csi_string.split(" ") if x != '']

        # Create list of imaginary and real numbers from CSI
      for i in range(len(csi_raw)):
          if i % 2 == 0:
              imaginary.append(csi_raw[i])
          else:
              real.append(csi_raw[i])

        # Transform imaginary and real into amplitude and phase
      for i in range(int(len(csi_raw) / 2)):
          amplitudes.append(sqrt(imaginary[i] ** 2 + real[i] ** 2))
      data.append(amplitudes)
    temp_df = pd.DataFrame(data)
    amp_dataframes.append(temp_df)



In [ ]:
print(len(amp_dataframes))
print(len(label_list))

In [ ]:
print(label_list)

In [ ]:
print(amp_dataframes[1].shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(amp_dataframes[2][50],color='g')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.title('Raw CSI Amplitude Data: LL')
plt.show()

In [ ]:
print(amp_dataframes[2][50].head(10))

## Apply noise filter

In [ ]:
denoised_dataframes = []
for amplitude in amp_dataframes:
    filtered_data = pd.DataFrame()
    for col in amplitude.columns:
      col_series = amplitude[col]
      # Hampel filter
      hampel_filtered = hampel(col_series, window_size=10)
      # Savitzky-Golay filter
      sg_filtered = savgol_filter(hampel_filtered.filtered_data, window_length=10, polyorder=3)
      filtered_data[col] = sg_filtered
    denoised_dataframes.append(filtered_data)


In [ ]:
print(len(denoised_dataframes))

In [ ]:
path = 'c:/Users/Ayush/Desktop/New folder (2)/ESP32-WiFi-Sensing-master/datasets/tvat-filtered-2/'
for i, dataframe in enumerate(denoised_dataframes):
  filename = f"{label_list[i]}-{i+1}.csv"
  dataframe.to_csv(path+filename, index=False)


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(denoised_dataframes[2][50],color='g')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.title('Denoise CSI Amplitude Data: LL')
plt.show()

In [ ]:
print(denoised_dataframes[2][50].head(10))

In [ ]:

columns_to_drop = [2,3,4,5,32,59,60,61,62,63]
for df in amp_dataframes:
    df.drop(df.columns[columns_to_drop], axis=1,inplace=True)


## Segmentation

In [ ]:
segment_dataframes = []
labels = []
for i, df in enumerate(amp_dataframes):
  df_len = len(df)
  #df['label'] = label_list[i]
  df.drop(df.columns[-1],axis=1,inplace=True)
  segment_len = (df_len//200)*200
  rows_to_skip = len(df) - segment_len
  rounded_df = df.iloc[rows_to_skip:]
  segment_df = np.array_split(rounded_df, range(200, len(rounded_df), 200))
  for segment in segment_df:
    segment_dataframes.append(segment)
    labels.append(label_list[i])


In [ ]:
print(len(segment_dataframes))
print(len(labels))

In [ ]:
print(segment_dataframes[3])

## CNN Model

In [ ]:
# Convert DataFrames to matrices
matrix_list = [df.values for df in segment_dataframes]

# Stack matrices vertically
stacked_matrix = np.vstack(matrix_list)

In [ ]:
dataset_array = np.concatenate(segment_dataframes, axis=0)

In [ ]:
print(matrix_list.shape)

In [ ]:
print(stacked_matrix)

In [ ]:
from sklearn.model_selection import train_test_split
labels = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(segment_dataframes, labels, test_size=0.3, random_state=42)

X_train = np.array(X_train)
X_train = X_train.astype('float32')
X_train /= 255
X_test = np.array(X_test)
X_test = X_test.astype('float32')
X_test /= 255

In [ ]:
print(X_train.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(y_train)
y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout,Attention
from tensorflow.keras.losses import SparseCategoricalCrossentropy


# Define the CNN model
model = keras.Sequential([
    keras.layers.Conv2D(32, 7, activation='relu' , input_shape=(200, 53,1)),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(96, 5, activation='relu'),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(7, activation='softmax')

])

# Compile the model
model.compile(optimizer= tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train_encoded, epochs=50, batch_size=8,validation_split=0.2)